In [1]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import time

import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.preprocessing import LabelEncoder

from numpy.linalg import inv

Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2
# DATASET SETUP
# ============================================================

DATASET_ROOT = "/content/drive/MyDrive/Cocoa/amini_dataset"

BASE_DIR = os.path.join(
    DATASET_ROOT,
    "dataset"
)

CSV_PATH_TRAIN = os.path.join(
    DATASET_ROOT,
    "Train.csv"
)

IMAGE_DIR_TRAIN = os.path.join(
    BASE_DIR,
    "images",
    "train"
)

SAVE_DIR = os.path.join(
    DATASET_ROOT,
    "run"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

RANDOM_STATE = 42

IMAGE_SIZE = 224

# ------------------------------------------------------------
# Read Training CSV
# ------------------------------------------------------------

df = pd.read_csv(
    CSV_PATH_TRAIN
)

# ------------------------------------------------------------
# Remove images belonging to multiple classes
# ------------------------------------------------------------

image_class_count = (
    df.groupby("Image_ID")["class"]
      .nunique()
)

ambiguous_images = set(

    image_class_count[
        image_class_count > 1
    ].index

)

print("=" * 60)
print("DATASET")
print("=" * 60)

print(
    "Total CSV Rows :",
    len(df)
)

print(
    "Unique Images :",
    df["Image_ID"].nunique()
)

print(
    "Ambiguous Images :",
    len(ambiguous_images)
)

# ------------------------------------------------------------
# Keep only clean images
# ------------------------------------------------------------

clean_df = df[
    ~df["Image_ID"].isin(
        ambiguous_images
    )
]

# ------------------------------------------------------------
# One row per image
# ------------------------------------------------------------

df = (

    clean_df[
        [
            "Image_ID",
            "class"
        ]
    ]

    .drop_duplicates()

    .reset_index(drop=True)

)

# ------------------------------------------------------------
# Add image path
# ------------------------------------------------------------

df["image_path"] = df["Image_ID"].apply(

    lambda x:

    os.path.join(
        IMAGE_DIR_TRAIN,
        x
    )

)

# ------------------------------------------------------------
# Class Information
# ------------------------------------------------------------

CLASS_NAMES = sorted(
    df["class"].unique()
)

print()

print("=" * 60)
print("CLASSES")
print("=" * 60)

print(CLASS_NAMES)

print()

print("Total Images :", len(df))

print()

print(
    df["class"]
      .value_counts()
)

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["class"]
)

NUM_CLASSES = len(
    label_encoder.classes_
)

print()

print("Label Mapping")

for i, cls in enumerate(
    label_encoder.classes_
):

    print(
        i,
        "->",
        cls
    )

# ------------------------------------------------------------
# Train / Validation Split
# ------------------------------------------------------------

df_train, df_val = train_test_split(

    df,

    test_size=0.20,

    stratify=df["label"],

    random_state=RANDOM_STATE

)

df_train = df_train.reset_index(
    drop=True
)

df_val = df_val.reset_index(
    drop=True
)

# ------------------------------------------------------------
# Verify No Leakage
# ------------------------------------------------------------

overlap = set(
    df_train["Image_ID"]
).intersection(
    set(
        df_val["Image_ID"]
    )
)

print()

print("=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

print(
    "Overlapping images:",
    len(overlap)
)

# ------------------------------------------------------------
# Split Summary
# ------------------------------------------------------------

print()

print("=" * 60)
print("TRAIN SPLIT")
print("=" * 60)

print(
    df_train["class"]
      .value_counts()
)

print()

print("=" * 60)
print("VALIDATION SPLIT")
print("=" * 60)

print(
    df_val["class"]
      .value_counts()
)

# ------------------------------------------------------------
# Save Splits
# ------------------------------------------------------------

TRAIN_SPLIT_PATH = os.path.join(

    SAVE_DIR,

    "train_split.csv"

)

VAL_SPLIT_PATH = os.path.join(

    SAVE_DIR,

    "validation_split.csv"

)

df_train.to_csv(

    TRAIN_SPLIT_PATH,

    index=False

)

df_val.to_csv(

    VAL_SPLIT_PATH,

    index=False

)

print()

print("Saved")

print(TRAIN_SPLIT_PATH)

print(VAL_SPLIT_PATH)

# ============================================================
# LOAD PRETRAINED MOBILENET
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

weights = MobileNet_V3_Small_Weights.DEFAULT

model = mobilenet_v3_small(
    weights=weights
)

print()

print("=" * 60)
print("PRETRAINED MOBILENET LOADED")
print("=" * 60)

print(model)

# ============================================================
# CONVERT TO FEATURE EXTRACTOR
# ============================================================

feature_extractor = nn.Sequential(

    model.features,

    model.avgpool,

    nn.Flatten()

).to(device)

feature_extractor.eval()

dummy = torch.randn(

    1,
    3,
    224,
    224

).to(device)

with torch.no_grad():

    output = feature_extractor(
        dummy
    )

print()

print("=" * 60)
print("FEATURE EXTRACTOR")
print("=" * 60)

print(
    "Feature Shape :",
    output.shape
)

DATASET
Total CSV Rows : 9792
Unique Images : 5529
Ambiguous Images : 6

CLASSES
['anthracnose', 'cssvd', 'healthy']

Total Images : 5523

class
cssvd          2237
healthy        1720
anthracnose    1566
Name: count, dtype: int64

Label Mapping
0 -> anthracnose
1 -> cssvd
2 -> healthy

LEAKAGE CHECK
Overlapping images: 0

TRAIN SPLIT
class
cssvd          1789
healthy        1376
anthracnose    1253
Name: count, dtype: int64

VALIDATION SPLIT
class
cssvd          448
healthy        344
anthracnose    313
Name: count, dtype: int64

Saved
/content/drive/MyDrive/Cocoa/amini_dataset/run/train_split.csv
/content/drive/MyDrive/Cocoa/amini_dataset/run/validation_split.csv
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 89.9MB/s]



PRETRAINED MOBILENET LOADED
MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d

In [3]:
train_ids = set(df_train["Image_ID"])
val_ids = set(df_val["Image_ID"])

overlap = train_ids.intersection(val_ids)

print("Overlap:", len(overlap))

Overlap: 0


In [4]:
# ============================================================
# CELL 3
# EXTRACT MOBILENET FEATURES
# ============================================================

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "validation_features.csv"
)

# ------------------------------------------------------------
# Image Transform
# ------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# Load Existing Features
# ------------------------------------------------------------

if (
    os.path.exists(TRAIN_FEATURE_PATH)
    and
    os.path.exists(VAL_FEATURE_PATH)
):

    print("=" * 60)
    print("FEATURE FILES FOUND")
    print("=" * 60)

    extracted_features_train = pd.read_csv(
        TRAIN_FEATURE_PATH
    )

    extracted_features_val = pd.read_csv(
        VAL_FEATURE_PATH
    )

    print(
        extracted_features_train.shape
    )

    print(
        extracted_features_val.shape
    )

else:

    # --------------------------------------------------------
    # Feature Extraction Function
    # --------------------------------------------------------

    def extract_features(dataframe):

        rows = []

        feature_extractor.eval()

        with torch.no_grad():

            for _, row in tqdm(
                dataframe.iterrows(),
                total=len(dataframe)
            ):

                image = Image.open(
                    row["image_path"]
                ).convert("RGB")

                image = transform(image)

                image = image.unsqueeze(0).to(device)

                features = feature_extractor(
                    image
                )

                features = (
                    features
                    .cpu()
                    .numpy()
                    .flatten()
                )

                rows.append([

                    row["Image_ID"],

                    row["class"],

                    *features

                ])

        columns = (

            ["Image_ID", "class"]

            +

            [
                f"feat_{i}"
                for i in range(576)
            ]

        )

        return pd.DataFrame(
            rows,
            columns=columns
        )

    # --------------------------------------------------------
    # Training Features
    # --------------------------------------------------------

    print()

    print("=" * 60)
    print("TRAIN FEATURES")
    print("=" * 60)

    extracted_features_train = extract_features(
        df_train
    )

    # --------------------------------------------------------
    # Validation Features
    # --------------------------------------------------------

    print()

    print("=" * 60)
    print("VALIDATION FEATURES")
    print("=" * 60)

    extracted_features_val = extract_features(
        df_val
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    extracted_features_train.to_csv(

        TRAIN_FEATURE_PATH,

        index=False

    )

    extracted_features_val.to_csv(

        VAL_FEATURE_PATH,

        index=False

    )

    print()

    print("Saved")

    print(TRAIN_FEATURE_PATH)

    print(VAL_FEATURE_PATH)

print()

print(extracted_features_train.head())

print()

print(extracted_features_train.shape)

print(extracted_features_val.shape)

FEATURE FILES FOUND
(4418, 578)
(1105, 578)

         Image_ID        class    feat_0    feat_1    feat_2    feat_3  \
0   ID_Kf4dHB.jpg      healthy  0.592689  0.262962  0.541031  0.204872   
1  ID_GK394N.jpeg      healthy  0.105462  0.183077 -0.087441 -0.140564   
2   ID_u21Vuy.jpg        cssvd  0.034122 -0.065241 -0.069107 -0.090574   
3   ID_n2PA4u.jpg  anthracnose  0.107449  0.013330 -0.015691  0.175711   
4   ID_A397lM.jpg        cssvd -0.098958 -0.125566 -0.003058 -0.093994   

     feat_4    feat_5    feat_6    feat_7  ...  feat_566  feat_567  feat_568  \
0  0.200211  0.185831  0.264065  0.654791  ...  0.273143  0.435094  0.015209   
1  1.870920 -0.099488  0.024553  0.035892  ...  0.039370  0.581388  0.168551   
2  0.162255  0.110419 -0.077386  0.081519  ...  0.703579  0.400401 -0.131235   
3  0.145163  0.022977  0.119661 -0.105688  ...  0.324440  0.501407 -0.043332   
4  2.268527  0.252645 -0.196929 -0.028630  ...  0.529999  0.968149 -0.189001   

   feat_569  feat_570  feat_5

In [5]:
!pip install -q onnx onnxscript onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 5.7 MB/s eta 0:00:00


In [6]:
# ============================================================
# EXPORT PRETRAINED MOBILENETV3-SMALL FEATURE EXTRACTOR
# ============================================================

import torch.onnx

ONNX_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_v3_small_features.onnx"
)

feature_extractor.eval()

dummy_input = torch.randn(
    1,
    3,
    224,
    224,
    device=device
)

torch.onnx.export(
    feature_extractor,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=17,
    input_names=["input"],
    output_names=["features"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "features": {0: "batch_size"}
    }
)

print("=" * 60)
print("MOBILENET EXPORTED")
print("=" * 60)

print("Saved to:")
print(ONNX_PATH)

/tmp/ipykernel_2997/3765644657.py:22: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0913 15:37:45.089000 2997 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:56: 

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
MOBILENET EXPORTED
Saved to:
/content/drive/MyDrive/Cocoa/amini_dataset/run/mobilenet_v3_small_features.onnx


In [7]:
# ============================================================
# VERIFY ONNX OUTPUT
# ============================================================

import onnx
import onnxruntime as ort

onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

test_input = np.random.randn(
    1, 3, 224, 224
).astype(np.float32)

onnx_output = session.run(
    ["features"],
    {"input": test_input}
)[0]

with torch.no_grad():
    pytorch_output = feature_extractor(
        torch.from_numpy(test_input).to(device)
    ).cpu().numpy()

print("PyTorch shape :", pytorch_output.shape)
print("ONNX shape   :", onnx_output.shape)

print(
    "Maximum difference:",
    np.max(
        np.abs(
            pytorch_output - onnx_output
        )
    )
)

PyTorch shape : (1, 576)
ONNX shape   : (1, 576)
Maximum difference: 2.6226044e-06


In [8]:
# ============================================================
# ONNX + EXISTING JACOBI-DMR EVALUATION
# ============================================================

import onnxruntime as ort
import numpy as np
import pandas as pd
import time
import json

from PIL import Image


# ============================================================
# LOAD ONNX MODEL
# ============================================================

ONNX_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_v3_small_features.onnx"
)

session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print("=" * 60)
print("ONNX MODEL")
print("=" * 60)

print("Input :", input_name)
print("Output:", output_name)


# ============================================================
# SAME PREPROCESSING AS ORIGINAL PYTORCH PIPELINE
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# LOAD EXISTING JACOBI COEFFICIENTS
# ============================================================

JACOBI_PATH = os.path.join(
    SAVE_DIR,
    "jacobi_coefficients.json"
)

with open(
    JACOBI_PATH,
    "r"
) as f:

    betas = json.load(f)

# Convert lists to numpy arrays
betas = {
    cls: np.array(beta, dtype=np.float32)
    for cls, beta in betas.items()
}

print()
print("=" * 60)
print("JACOBI MODEL")
print("=" * 60)

print("Classes:", list(betas.keys()))

for cls, beta in betas.items():

    print(
        f"{cls}: {beta.shape}"
    )


# ============================================================
# EXTRACT ONNX FEATURES
# ============================================================

def extract_onnx_features(dataframe):

    rows = []

    start = time.time()

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe)
    ):

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        image = transform(image)

        # [3,224,224]
        # →
        # [1,3,224,224]

        image = (
            image
            .unsqueeze(0)
            .numpy()
            .astype(np.float32)
        )

        features = session.run(
            [output_name],
            {
                input_name: image
            }
        )[0]

        features = features.flatten()

        rows.append([
            row["Image_ID"],
            row["class"],
            *features
        ])

    elapsed = (
        time.time() -
        start
    )

    columns = (
        ["Image_ID", "class"]
        +
        [
            f"feat_{i}"
            for i in range(576)
        ]
    )

    result = pd.DataFrame(
        rows,
        columns=columns
    )

    return result, elapsed


# ============================================================
# VALIDATION FEATURES ONLY
# ============================================================

print()
print("=" * 60)
print("ONNX VALIDATION FEATURE EXTRACTION")
print("=" * 60)

onnx_val_features, onnx_feature_time = (
    extract_onnx_features(df_val)
)

print()
print(
    "Feature shape:",
    onnx_val_features.shape
)

print(
    f"Feature extraction time: "
    f"{onnx_feature_time:.4f}s"
)


# ============================================================
# PREPARE VALIDATION FEATURES
# ============================================================

X_val_onnx = (
    onnx_val_features
    .drop(
        columns=[
            "Image_ID",
            "class"
        ]
    )
    .values
)

y_true_onnx = (
    onnx_val_features["class"]
)


# ============================================================
# JACOBI-DMR PREDICTION
# USING EXISTING COEFFICIENTS
# ============================================================

time_jacobi_start = time.time()

lambda_scores_onnx = {}

for cls in betas:

    lambda_scores_onnx[cls] = np.exp(
        X_val_onnx @ betas[cls]
    )

lambda_df_onnx = pd.DataFrame(
    lambda_scores_onnx
)

y_pred_onnx = (
    lambda_df_onnx
    .idxmax(axis=1)
)


time_jacobi_onnx = (
    time.time() -
    time_jacobi_start
)


# ============================================================
# EVALUATION
# ============================================================

acc_onnx = accuracy_score(
    y_true_onnx,
    y_pred_onnx
)

cm_onnx = confusion_matrix(
    y_true_onnx,
    y_pred_onnx,
    labels=CLASS_NAMES
)


print()
print("=" * 60)
print("ONNX + EXISTING JACOBI-DMR")
print("=" * 60)

print(
    f"Accuracy: {acc_onnx:.4f}"
)

print(
    f"Feature extraction time: "
    f"{onnx_feature_time:.4f}s"
)

print(
    f"Jacobi prediction time: "
    f"{time_jacobi_onnx:.4f}s"
)

print(
    f"Total time: "
    f"{onnx_feature_time + time_jacobi_onnx:.4f}s"
)

print()

print(
    classification_report(
        y_true_onnx,
        y_pred_onnx,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:")
print(cm_onnx)


# ============================================================
# COMPARE WITH ORIGINAL PYTORCH PREDICTIONS
# ============================================================

# Original predictions from the notebook
X_val_original = (
    extracted_features_val
    .drop(
        columns=[
            "Image_ID",
            "class"
        ]
    )
    .values
)

lambda_scores_original = {}

for cls in betas:

    lambda_scores_original[cls] = np.exp(
        X_val_original @ betas[cls]
    )

lambda_df_original = pd.DataFrame(
    lambda_scores_original
)

y_pred_original = (
    lambda_df_original
    .idxmax(axis=1)
)


# ============================================================
# PYTORCH vs ONNX AGREEMENT
# ============================================================

prediction_match = (
    y_pred_original.values ==
    y_pred_onnx.values
)

num_matches = prediction_match.sum()

num_total = len(
    prediction_match
)

agreement = (
    num_matches /
    num_total
)

print()
print("=" * 60)
print("PYTORCH vs ONNX")
print("=" * 60)

print(
    f"Matching predictions: "
    f"{num_matches}/{num_total}"
)

print(
    f"Prediction agreement: "
    f"{agreement:.4%}"
)

ONNX MODEL
Input : input
Output: features

JACOBI MODEL
Classes: ['anthracnose', 'cssvd', 'healthy']
anthracnose: (576,)
cssvd: (576,)
healthy: (576,)

ONNX VALIDATION FEATURE EXTRACTION


100%|██████████| 1105/1105 [12:52<00:00,  1.43it/s]



Feature shape: (1105, 578)
Feature extraction time: 772.6174s

ONNX + EXISTING JACOBI-DMR
Accuracy: 0.8127
Feature extraction time: 772.6174s
Jacobi prediction time: 0.0108s
Total time: 772.6283s

              precision    recall  f1-score   support

 anthracnose       0.84      0.81      0.83       313
       cssvd       0.82      0.81      0.82       448
     healthy       0.77      0.82      0.79       344

    accuracy                           0.81      1105
   macro avg       0.81      0.81      0.81      1105
weighted avg       0.81      0.81      0.81      1105


Confusion Matrix:
[[255  33  25]
 [ 29 362  57]
 [ 19  44 281]]

PYTORCH vs ONNX
Matching predictions: 1105/1105
Prediction agreement: 100.0000%


In [9]:
# ============================================================
# SAVE CLASS NAMES
# ============================================================

CLASS_NAMES_PATH = os.path.join(
    SAVE_DIR,
    "class_names.json"
)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(
        CLASS_NAMES,
        f,
        indent=2
    )

print("Class names saved to:")
print(CLASS_NAMES_PATH)

print()
print("Classes:")
print(CLASS_NAMES)

Class names saved to:
/content/drive/MyDrive/Cocoa/amini_dataset/run/class_names.json

Classes:
['anthracnose', 'cssvd', 'healthy']


In [10]:
# ============================================================
# CELL 4
# TRAIN JACOBI-DMR
# ============================================================

import time

time_jacobi_start = time.time()

# ------------------------------------------------------------
# Prepare Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_train = extracted_features_train["class"]

X_train = X_train.values

# ------------------------------------------------------------
# One-Hot Encoding
# ------------------------------------------------------------

y_one_hot = pd.get_dummies(
    y_train
)

# ------------------------------------------------------------
# Jacobi Hyperparameters
# ------------------------------------------------------------

a = b = 1 / len(X_train)

k = 1

# ------------------------------------------------------------
# Train One Classifier Per Class
# ------------------------------------------------------------

betas = {}

for cls in y_one_hot.columns:

    eta = np.log(

        (y_one_hot[cls] + a)

        /

        (1 + k * b)

    )

    beta = inv(

        X_train.T @ X_train

    ) @ (

        X_train.T @ eta

    )

    betas[cls] = beta

# ------------------------------------------------------------
# Prepare Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

X_val = X_val.values

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

lambda_scores = {}

for cls in betas:

    lambda_scores[cls] = np.exp(
        X_val @ betas[cls]
    )

lambda_df = pd.DataFrame(
    lambda_scores
)

y_pred = lambda_df.idxmax(
    axis=1
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

time_jacobi_seconds = (
    time.time() -
    time_jacobi_start
)

acc_jacobi = accuracy_score(
    y_true,
    y_pred
)

cm_jacobi = confusion_matrix(
    y_true,
    y_pred,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("JACOBI-DMR")
print("=" * 60)

print(
    f"Accuracy: {acc_jacobi:.4f}"
)

print(
    f"Time: {time_jacobi_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_jacobi)

# ------------------------------------------------------------
# Save Coefficients
# ------------------------------------------------------------

JACOBI_PATH = os.path.join(
    SAVE_DIR,
    "jacobi_coefficients.json"
)

with open(
    JACOBI_PATH,
    "w"
) as f:

    json.dump(
        {
            cls: betas[cls].tolist()
            for cls in betas
        },
        f
    )

print()

print("Jacobi coefficients saved to:")

print(JACOBI_PATH)


JACOBI-DMR
Accuracy: 0.8127
Time: 0.3487s

              precision    recall  f1-score   support

 anthracnose       0.84      0.81      0.83       313
       cssvd       0.82      0.81      0.82       448
     healthy       0.77      0.82      0.79       344

    accuracy                           0.81      1105
   macro avg       0.81      0.81      0.81      1105
weighted avg       0.81      0.81      0.81      1105


Confusion Matrix:

[[255  33  25]
 [ 29 362  57]
 [ 19  44 281]]

Jacobi coefficients saved to:
/content/drive/MyDrive/Cocoa/amini_dataset/run/jacobi_coefficients.json


In [11]:
# ============================================================
# RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

import time

time_rf_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_rf = rf_model.predict(
    X_val
)

time_rf_seconds = (
    time.time() -
    time_rf_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_rf = accuracy_score(
    y_true,
    y_pred_rf
)

cm_rf = confusion_matrix(
    y_true,
    y_pred_rf,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("RANDOM FOREST")
print("=" * 60)

print(
    f"Accuracy: {acc_rf:.4f}"
)

print(
    f"Time: {time_rf_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_rf,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_rf)


RANDOM FOREST
Accuracy: 0.7620
Time: 16.9197s

              precision    recall  f1-score   support

 anthracnose       0.82      0.66      0.73       313
       cssvd       0.76      0.82      0.79       448
     healthy       0.73      0.78      0.75       344

    accuracy                           0.76      1105
   macro avg       0.77      0.75      0.76      1105
weighted avg       0.77      0.76      0.76      1105


Confusion Matrix:

[[207  60  46]
 [ 28 367  53]
 [ 17  59 268]]


In [12]:
# ============================================================
# SUPPORT VECTOR MACHINE (SVM)
# ============================================================

from sklearn.svm import SVC

import time

time_svm_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    random_state=42
)

svm_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_svm = svm_model.predict(
    X_val
)

time_svm_seconds = (
    time.time() -
    time_svm_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_svm = accuracy_score(
    y_true,
    y_pred_svm
)

cm_svm = confusion_matrix(
    y_true,
    y_pred_svm,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("SUPPORT VECTOR MACHINE")
print("=" * 60)

print(
    f"Accuracy: {acc_svm:.4f}"
)

print(
    f"Time: {time_svm_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_svm,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_svm)


SUPPORT VECTOR MACHINE
Accuracy: 0.8353
Time: 9.0725s

              precision    recall  f1-score   support

 anthracnose       0.86      0.79      0.82       313
       cssvd       0.85      0.85      0.85       448
     healthy       0.80      0.86      0.83       344

    accuracy                           0.84      1105
   macro avg       0.84      0.83      0.83      1105
weighted avg       0.84      0.84      0.84      1105


Confusion Matrix:

[[248  32  33]
 [ 27 380  41]
 [ 14  35 295]]


In [13]:
# ============================================================
# RIDGE CLASSIFIER
# ============================================================

from sklearn.linear_model import RidgeClassifier

import time

time_ridge_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

ridge_model = RidgeClassifier(
    alpha=1.0,
    random_state=42
)

ridge_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_ridge = ridge_model.predict(
    X_val
)

time_ridge_seconds = (
    time.time() -
    time_ridge_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_ridge = accuracy_score(
    y_true,
    y_pred_ridge
)

cm_ridge = confusion_matrix(
    y_true,
    y_pred_ridge,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("RIDGE CLASSIFIER")
print("=" * 60)

print(
    f"Accuracy: {acc_ridge:.4f}"
)

print(
    f"Time: {time_ridge_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_ridge,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_ridge)


RIDGE CLASSIFIER
Accuracy: 0.8190
Time: 0.3016s

              precision    recall  f1-score   support

 anthracnose       0.85      0.81      0.83       313
       cssvd       0.83      0.81      0.82       448
     healthy       0.78      0.83      0.80       344

    accuracy                           0.82      1105
   macro avg       0.82      0.82      0.82      1105
weighted avg       0.82      0.82      0.82      1105


Confusion Matrix:

[[254  33  26]
 [ 26 365  57]
 [ 18  40 286]]


In [14]:
# ============================================================
# LOGISTIC REGRESSION (L1 / LASSO)
# ============================================================

from sklearn.linear_model import LogisticRegression

import time

time_lasso_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

lasso_model = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=1.0,
    max_iter=5000,
    random_state=42
)

lasso_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_lasso = lasso_model.predict(
    X_val
)

time_lasso_seconds = (
    time.time() -
    time_lasso_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_lasso = accuracy_score(
    y_true,
    y_pred_lasso
)

cm_lasso = confusion_matrix(
    y_true,
    y_pred_lasso,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("LOGISTIC REGRESSION (L1 / LASSO)")
print("=" * 60)

print(
    f"Accuracy: {acc_lasso:.4f}"
)

print(
    f"Time: {time_lasso_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_lasso,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_lasso)


LOGISTIC REGRESSION (L1 / LASSO)
Accuracy: 0.8145
Time: 91.1946s

              precision    recall  f1-score   support

 anthracnose       0.82      0.83      0.83       313
       cssvd       0.83      0.81      0.82       448
     healthy       0.78      0.81      0.80       344

    accuracy                           0.81      1105
   macro avg       0.81      0.82      0.81      1105
weighted avg       0.82      0.81      0.81      1105


Confusion Matrix:

[[259  31  23]
 [ 32 362  54]
 [ 23  42 279]]


In [15]:
# ============================================================
# FINAL MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({

    "Model": [

        "Jacobi-DMR",
        "Random Forest",
        "Support Vector Machine",
        "Ridge Classifier",
        "Logistic Regression (L1)"

    ],

    "Accuracy": [

        acc_jacobi,
        acc_rf,
        acc_svm,
        acc_ridge,
        acc_lasso

    ],

    "Execution Time (s)": [

        time_jacobi_seconds,
        time_rf_seconds,
        time_svm_seconds,
        time_ridge_seconds,
        time_lasso_seconds

    ]

})

comparison_df["Accuracy"] = comparison_df["Accuracy"].round(4)

comparison_df["Execution Time (s)"] = comparison_df[
    "Execution Time (s)"
].round(4)

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print()

print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(comparison_df)

comparison_path = os.path.join(
    SAVE_DIR,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print()

print(f"Comparison saved to:\n{comparison_path}")


MODEL COMPARISON
                      Model  Accuracy  Execution Time (s)
0    Support Vector Machine    0.8353              9.0725
1          Ridge Classifier    0.8190              0.3016
2  Logistic Regression (L1)    0.8145             91.1946
3                Jacobi-DMR    0.8127              0.3487
4             Random Forest    0.7620             16.9197

Comparison saved to:
/content/drive/MyDrive/Cocoa/amini_dataset/run/model_comparison.csv
